### Python Memory Management - Step by Step Guide

# 1. Understanding Memory Management in Python

## 1.1 Variables and Memory References
In Python, variables are references to objects stored in memory rather than the actual data itself.

In [ ]:
x = 42

y = x

print(id(x))  # Memory address of object referenced by x
print(id(y))  # Memory address of object referenced by y


The function `id()` returns the memory address of the object as a base-10 number.

# 2. Reference Counting
Python uses reference counting to manage memory. An object is deallocated when its reference count drops to zero.  
References can be counted with built-in function `sys.getrefcount()`.

In [ ]:
import sys

x = [1, 2, 3]
print(sys.getrefcount(x))  # Get reference count for object x

y = x  # Assign another reference to the same object
print(sys.getrefcount(x))  # Reference count increases

del y  # Removing a reference
print(sys.getrefcount(x))  # Reference count decreases


The `getrefcount()` function takes object as an argument and creates temporary reference to it during the function call, hence the count starts from 2.

Another way to get reference count is by using `ctypes` library.

In [ ]:
import ctypes

address = id(x) # Memory address of x
print(ctypes.c_long.from_address(address).value) # Get reference count for object x 


The difference from `getrefcount()` function is that `ctypes.c_long.from_address()` directly reads the memory at a given address, without affecting Python's reference count.

For more information on `ctypes` library:  
https://docs.python.org/3/library/ctypes.html

# 3. Garbage Collection
Python uses automatic garbage collection to remove objects no longer in use.

Sometimes, manual garbage collection is useful for breaking reference cycles, optimizing memory in long-running programs, and debugging memory leaks, but should be used sparingly to avoid performance overhead.

In [7]:
import gc

class Test:
    def __del__(self):  # Destructor method called when object is deleted
        print("Object destroyed")

obj = Test()

del obj  # Explicitly deleting the object

collected = gc.collect()  # Run garbage collection manually
print(f"Garbage collector: collected {collected} objects")



Object destroyed
Garbage collector: collected 6 objects


# 4. Memory Leaks and Circular References

## 4.1 Circular References Leading to Memory Leaks

A memory leak occurs when objects are no longer needed but are not released due to lingering references, such as *circular references*.  
Python’s garbage collector can detect and clean up circular references, but in some cases, manual intervention is needed.


In [4]:
import gc
import sys

class CircularReference:
    def __init__(self):
        self.reference = self  # Creates a circular reference

obj = CircularReference()

print(f"Number of references for obj: {sys.getrefcount(obj)}")

del obj  # Even after deletion, the object is not garbage collected

collected = gc.collect()
print(f"Garbage collector: collected {collected} objects")


Number of references for obj: 3
Garbage collector: collected 7 objects


## 4.2 Weak References

To avoid circular reference issues, use *weak references* (`weakref` module) that do not increase reference counts.


In [ ]:
import weakref

class B:
    pass

obj_b = B()
weak_ref = weakref.ref(obj_b)  # Create a weak reference

print(weak_ref())  # Output: <__main__.B object at ...>

del obj_b
print(weak_ref())  # Output: None (object has been garbage collected)


None


Using weak references ensures that objects do not persist beyond their needed lifetime, preventing memory leaks.